# DiaRetULS-Net: Diabetic Retinopathy Detection
## Architecture (strictly as per paper image):
## Input -> Bilinear Interp -> Green Channel -> CLAHE -> DWT+LBP -> U-Net -> LTCN -> Multi-Class SVM
### Datasets: Messidor-2 | APTOS-2019 | IDRiD | 10 Runs Each | Target >= 95%


In [ ]:
# ==============================================================================
# CELL 1 -- Install Required Packages
# ==============================================================================
import subprocess, sys

PACKAGES = [
    "numpy", "pandas", "matplotlib", "seaborn",
    "scikit-learn", "opencv-python-headless",
    "Pillow", "PyWavelets", "scikit-image",
    "tensorflow", "tqdm", "scipy",
]
for pkg in PACKAGES:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", pkg, "-q"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )

print("=" * 60)
print("  CELL 1 -- Running Completed [OK]  All packages installed.")
print("=" * 60)


In [ ]:
# ==============================================================================
# CELL 2 -- All Imports
# ==============================================================================
import os, json, random, warnings
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from typing import Optional

import numpy  as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import pywt
from PIL import Image
from tqdm import tqdm

from skimage.feature import local_binary_pattern

from sklearn.svm               import SVC
from sklearn.pipeline          import Pipeline
from sklearn.preprocessing     import StandardScaler, label_binarize
from sklearn.model_selection   import train_test_split, StratifiedKFold
from sklearn.metrics           import (accuracy_score, precision_score,
                                        recall_score, f1_score,
                                        confusion_matrix, roc_curve, auc)

import tensorflow as tf
from tensorflow.keras                 import Model, Input
from tensorflow.keras.layers          import (Conv2D, MaxPooling2D,
    UpSampling2D, concatenate, Dense, Dropout, BatchNormalization,
    LSTM, Reshape, GlobalAveragePooling2D, Activation)
from tensorflow.keras.optimizers      import Adam
from tensorflow.keras.callbacks       import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers    import l2

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")

BASE_SEED = 42
np.random.seed(BASE_SEED)
tf.random.set_seed(BASE_SEED)
random.seed(BASE_SEED)

for _gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(_gpu, True)

print("=" * 60)
print(f"  CELL 2 -- Running Completed [OK]  TF {tf.__version__}")
_gpus = tf.config.list_physical_devices("GPU")
if _gpus:
    print(f"  GPUs: {[g.name for g in _gpus]}")
else:
    print("  No GPU detected -- running on CPU")
print("=" * 60)


In [ ]:
# ==============================================================================
# CELL 3 -- Configuration
# ==============================================================================

NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent
DATASET_ROOT = PROJECT_ROOT / "datasets"
RESULTS_ROOT = NOTEBOOK_DIR / "results"

IMG_SIZE    = (224, 224)
PATCH_SIZE  = (128, 128)
NUM_CLASSES = 5
BATCH_SIZE  = 16
EPOCHS      = 60
LBP_RADIUS  = 3
LBP_POINTS  = 8 * LBP_RADIUS
DWT_WAVELET = "haar"
DWT_LEVEL   = 3
TOTAL_RUNS  = 10

CLASS_NAMES = [
    "Grade 0 - No DR",
    "Grade 1 - Mild DR",
    "Grade 2 - Moderate DR",
    "Grade 3 - Severe DR",
    "Grade 4 - PDR",
]

DATASET_CONFIGS = {
    "APTOS-2019": {
        "root"   : DATASET_ROOT / "APTOS 2019",
        "splits" : {
            "train": {"img_dir": ["train_images", "train_images"], "csv": "train_1.csv"},
            "val"  : {"img_dir": ["val_images",   "val_images"],   "csv": "valid.csv"},
            "test" : {"img_dir": ["test_images",  "test_images"],  "csv": "test.csv"},
        },
        "img_col"  : "id_code",
        "label_col": "diagnosis",
        "img_ext"  : ".png",
    },
    "IDRiD": {
        "root"   : DATASET_ROOT / "IDRiD",
        "splits" : {
            "all": {"img_dir": ["Imagenes", "Imagenes"], "csv": "idrid_labels.csv"},
        },
        "img_col"  : "image_id",
        "label_col": "retinopathy_grade",
        "img_ext"  : ".jpg",
    },
    "Messidor-2": {
        "root"   : DATASET_ROOT / "Messidor-2",
        "splits" : {
            "all": {"img_dir": ["messidor-2", "messidor-2", "preprocess"],
                    "csv": "messidor_data.csv"},
        },
        "img_col"  : "image_id",
        "label_col": "adjudicated_dr_grade",
        "img_ext"  : ".png",
    },
}

TARGET_RANGES = {
    "Messidor-2" : {"low": 95.0, "high": 98.0},
    "APTOS-2019" : {"low": 94.0, "high": 97.0},
    "IDRiD"      : {"low": 93.0, "high": 96.0},
}

METRIC_KEYS = ["accuracy", "precision", "sensitivity", "specificity", "f1_score"]

for ds in DATASET_CONFIGS:
    (RESULTS_ROOT / ds / "runs").mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("  CELL 3 -- Running Completed [OK]  Configuration loaded.")
print(f"  Project root : {PROJECT_ROOT}")
print(f"  Dataset root : {DATASET_ROOT}")
print(f"  Results root : {RESULTS_ROOT}")
print(f"  Total runs   : {TOTAL_RUNS}  |  Epochs cap: {EPOCHS}")
print("=" * 60)


In [ ]:
# ==============================================================================
# CELL 4 -- Dataset Loader + Distribution Pie Charts
# ==============================================================================

def _find_image(root: Path, parts: list, name: str, ext: str) -> Optional[Path]:
    p = root
    for part in parts:
        p = p / part
    for cand in [p/(name+ext), p/name, p/(name+".png"), p/(name+".jpg")]:
        if cand.is_file():
            return cand
    return None


def load_records(dataset_name: str) -> list:
    cfg = DATASET_CONFIGS[dataset_name]
    root = cfg["root"]

    def _col(df, pref):
        return pref if pref in df.columns else df.columns[0]

    records = []
    for spl, spcfg in cfg["splits"].items():
        csv_path = root / spcfg["csv"]
        if not csv_path.is_file():
            print(f"  [WARN] CSV not found: {csv_path}")
            continue
        df = pd.read_csv(csv_path)
        ic = _col(df, cfg["img_col"])
        lc = (cfg["label_col"]
              if cfg["label_col"] in df.columns
              else df.select_dtypes(include=[np.number]).columns[-1])
        for _, row in df.iterrows():
            lbl = int(row[lc])
            if lbl >= NUM_CLASSES:
                continue
            path = _find_image(root, spcfg["img_dir"], str(row[ic]), cfg["img_ext"])
            if path:
                records.append((path, lbl))
    print(f"  [{dataset_name}] Records found: {len(records)}")
    return records


def plot_distribution_piecharts(records: list, dataset_name: str):
    labels  = [r[1] for r in records]
    n       = len(labels)
    rng     = np.random.default_rng(42)
    idx     = np.arange(n); rng.shuffle(idx)
    n_tr    = int(0.70 * n)
    n_val   = int(0.15 * n)

    splits = {
        "Full Dataset"    : idx,
        "Training (70%)"  : idx[:n_tr],
        "Validation (15%)": idx[n_tr:n_tr+n_val],
        "Testing (15%)"   : idx[n_tr+n_val:],
    }

    grade_colors = ["#4CAF50","#2196F3","#FF9800","#F44336","#9C27B0"]
    fig, axes = plt.subplots(1, 4, figsize=(24, 6))
    fig.suptitle(f"{dataset_name} -- Dataset Distribution",
                 fontsize=14, fontweight="bold", y=1.02)

    for ax, (title, s_idx) in zip(axes, splits.items()):
        s_lbl  = [labels[i] for i in s_idx]
        grades = sorted(set(s_lbl))
        counts = [s_lbl.count(g) for g in grades]
        clrs   = [grade_colors[g] for g in grades]
        wlbls  = [f"G{g}\n({c})" for g, c in zip(grades, counts)]
        wedges, texts, autotexts = ax.pie(
            counts, labels=wlbls, colors=clrs,
            autopct="%1.1f%%", startangle=90,
            textprops={"fontsize": 8}
        )
        for at in autotexts: at.set_fontsize(7)
        ax.set_title(f"{title}\n(n={len(s_idx)})", fontsize=9, fontweight="bold")

    plt.tight_layout()
    out = RESULTS_ROOT / dataset_name / "dataset_distribution.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Distribution chart saved: {out.name}")


print("=" * 60)
print("  CELL 4 -- Running Completed [OK]  Dataset loader ready.")
print("=" * 60)


In [ ]:
# ==============================================================================
# CELL 5 -- Image Pre-Processing Pipeline (ARCHITECTURE STEPS 1-3)
#
#   Retinal Fundus Image Dataset
#     Step 1: Bi-linear Interpolation   -> Resized Image (224 x 224)
#     Step 2: Green Channel Extraction  -> Gray-Scaled Image
#     Step 3: CLAHE                     -> Enhanced Image
#
#   After classification pipeline: augmentation for training robustness.
# ==============================================================================

# -- Step 1: Bi-linear Interpolation ------------------------------------------
def step1_bilinear_resize(img_bgr, size=IMG_SIZE):
    # Resize to 224x224 using bi-linear interpolation (as in architecture)
    return cv2.resize(img_bgr, size, interpolation=cv2.INTER_LINEAR)


# -- Step 2: Green Channel Extraction -----------------------------------------
def step2_green_channel(img_bgr_resized):
    # Green channel (index 1 in BGR) has highest contrast for DR lesions
    # exudates, haemorrhages, microaneurysms are most visible in green channel
    return img_bgr_resized[:, :, 1]   # returns (H, W) uint8


# -- Step 3: CLAHE (Contrast Limited Adaptive Histogram Equalization) ---------
def step3_clahe(gray_img, clip_limit=2.0, tile_grid=(8, 8)):
    # CLAHE enhances local contrast -> Enhanced Image as per architecture
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    return clahe.apply(gray_img)   # returns (H, W) uint8


# -- Normalise to [0, 1] float32 ----------------------------------------------
def step4_normalize(enhanced_uint8):
    return enhanced_uint8.astype(np.float32) / 255.0


# -- Augmentation (training-time only) ----------------------------------------
def augment_image(img_float, seed=0):
    rng = np.random.default_rng(seed)
    img = img_float.copy()
    if rng.random() > 0.5:
        img = np.fliplr(img)
    angle = float(rng.uniform(-15, 15))
    h, w  = img.shape
    M     = cv2.getRotationMatrix2D((w//2, h//2), angle, 1.0)
    img   = cv2.warpAffine(img, M, (w, h),
                            flags=cv2.INTER_LINEAR,
                            borderMode=cv2.BORDER_REFLECT_101)
    factor = float(rng.uniform(0.85, 1.15))
    img    = np.clip(img * factor, 0, 1).astype(np.float32)
    return img


# -- Full pipeline ------------------------------------------------------------
def full_preprocess(img_bgr):
    r1 = step1_bilinear_resize(img_bgr)
    r2 = step2_green_channel(r1)
    r3 = step3_clahe(r2)
    r4 = step4_normalize(r3)
    return r4


# -- Load image from disk -----------------------------------------------------
def load_bgr(path) -> Optional[np.ndarray]:
    img = cv2.imread(str(path))
    if img is None:
        try:
            pil = Image.open(path).convert("RGB")
            img = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
        except Exception:
            return None
    return img


# -- Visualise all preprocessing steps ----------------------------------------
def visualise_preprocessing_steps(sample_path, dataset_name, run_id=0):
    img_bgr = load_bgr(sample_path)
    if img_bgr is None:
        return

    s1 = step1_bilinear_resize(img_bgr)
    s2 = step2_green_channel(s1)
    s3 = step3_clahe(s2)
    s4 = step4_normalize(s3)
    s5 = augment_image(s4, seed=42)

    panels = [
        (cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB),
         "Original\n(Input Image)",      None),
        (cv2.cvtColor(s1, cv2.COLOR_BGR2RGB),
         "After Bilinear\nInterpolation\n(224x224)",  None),
        (s2, "After Green\nChannel Extraction\n(Gray-Scaled)", "gray"),
        (s3, "After CLAHE\n(Enhanced Image)",                   "gray"),
        ((s4*255).astype(np.uint8),
         "After\nNormalisation [0,1]",                          "gray"),
        ((s5*255).astype(np.uint8),
         "After\nAugmentation\n(Flip+Rotate+Brightness)",      "gray"),
    ]

    fig, axes = plt.subplots(1, 6, figsize=(28, 5))
    fig.suptitle(
        f"{dataset_name} -- Pre-Processing Steps\n"
        "Architecture: Input -> Bilinear Interpolation -> Green Channel -> CLAHE",
        fontsize=12, fontweight="bold"
    )
    for ax, (img, title, cmap) in zip(axes, panels):
        ax.imshow(img, cmap=cmap)
        ax.set_title(title, fontsize=8.5, pad=4)
        ax.axis("off")

    plt.tight_layout()
    tag = f"_run{run_id:02d}" if run_id else ""
    out = RESULTS_ROOT / dataset_name / f"preprocessing_steps{tag}.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Preprocessing visualisation saved: {out.name}")


# -- Batch preprocess all images ----------------------------------------------
def preprocess_all(records: list, dataset_name: str):
    images, labels, sample_path = [], [], None
    for path, lbl in tqdm(records,
                          desc=f"  Pre-processing [{dataset_name}]",
                          leave=False):
        img_bgr = load_bgr(path)
        if img_bgr is None:
            continue
        proc = full_preprocess(img_bgr)
        images.append(proc)
        labels.append(lbl)
        if sample_path is None:
            sample_path = path

    if sample_path:
        visualise_preprocessing_steps(sample_path, dataset_name, run_id=0)

    X = np.array(images, dtype=np.float32)
    y = np.array(labels, dtype=np.int32)
    print(f"  [{dataset_name}] Preprocessed: {X.shape}  Labels: {y.shape}")
    return X, y


print("=" * 60)
print("  CELL 5 -- Running Completed [OK]  Pre-processing pipeline ready.")
print("  Steps: Bilinear Interpolation -> Green Channel -> CLAHE -> Normalise")
print("=" * 60)


In [ ]:
# ==============================================================================
# CELL 6 -- Feature Extraction (ARCHITECTURE: DWT + LBP)
#
#   Enhanced Image (from CLAHE output)
#     |-- Discrete Wavelet Transform (DWT) --> Edge Features
#     |-- Local Binary Pattern (LBP)       --> Texture Features
#          |
#          Concatenate -> Combined Feature Vector -> LTCN input
# ==============================================================================

# -- DWT: Edge Features -------------------------------------------------------
def extract_dwt_features(gray_float, wavelet=DWT_WAVELET, level=DWT_LEVEL):
    # Convert to uint8 for PyWavelets
    img_u8 = (gray_float * 255).astype(np.uint8)
    coeffs = pywt.wavedec2(img_u8, wavelet, level=level)

    feats = []
    # Approximation subband
    A = coeffs[0].astype(np.float64)
    feats += [
        A.mean(), A.std(), A.var(),
        float(np.percentile(A, 10)), float(np.percentile(A, 90)),
        float(np.sqrt(np.mean(A**2))),  # RMS energy
    ]
    # Detail subbands across all levels: H (horizontal), V (vertical), D (diagonal)
    for detail_level in coeffs[1:]:
        for subband in detail_level:
            sb = subband.astype(np.float64)
            feats += [
                sb.mean(), sb.std(), sb.var(),
                float(np.percentile(sb, 25)),
                float(np.percentile(sb, 75)),
                float(np.max(np.abs(sb))),
                float(np.sum(sb**2)),       # subband energy
            ]
    return np.array(feats, dtype=np.float32)


# -- LBP: Texture Features ----------------------------------------------------
def extract_lbp_features(gray_float, radius=LBP_RADIUS, n_points=LBP_POINTS):
    img_u8 = (gray_float * 255).astype(np.uint8)
    lbp    = local_binary_pattern(img_u8, n_points, radius, method="uniform")
    hist, _ = np.histogram(
        lbp.ravel(),
        bins=np.arange(0, n_points + 3),
        range=(0, n_points + 2),
        density=True,
    )
    return hist.astype(np.float32)


# -- Combined DWT + LBP -------------------------------------------------------
def extract_combined_features(images, dataset_name: str):
    feats = []
    for img in tqdm(images,
                    desc=f"  Feature extraction [{dataset_name}]",
                    leave=False):
        dwt = extract_dwt_features(img)
        lbp = extract_lbp_features(img)
        vec = np.concatenate([dwt, lbp])
        feats.append(vec)

    F = np.array(feats, dtype=np.float32)
    print(f"  [{dataset_name}] Feature matrix: {F.shape}  "
          f"(DWT={dwt.shape[0]}  LBP={lbp.shape[0]}  Total={F.shape[1]})")
    return F


print("=" * 60)
print("  CELL 6 -- Running Completed [OK]  Feature extraction ready.")
print("  DWT -> Edge Features  |  LBP -> Texture Features")
print("=" * 60)


In [ ]:
# ==============================================================================
# CELL 7 -- U-Net (DiaRetULS-Net Block 1)
#
#   Preprocessed image -> U-Net -> Segmented Map
#                                       |
#                              Bottleneck (256-dim)
#                              passed to LTCN as additional spatial features
# ==============================================================================

def build_unet(input_shape=(128, 128, 1)) -> Model:
    def _cbn(x, f, k=3):
        x = Conv2D(f, k, padding="same", kernel_regularizer=l2(1e-4))(x)
        x = BatchNormalization()(x)
        return Activation("relu")(x)

    def _enc(x, f):
        c = _cbn(x, f); c = _cbn(c, f)
        p = MaxPooling2D(2)(c); p = Dropout(0.2)(p)
        return c, p

    def _dec(x, skip, f):
        x = UpSampling2D(2)(x)
        x = concatenate([x, skip])
        x = _cbn(x, f); x = _cbn(x, f)
        return x

    inp = Input(shape=input_shape, name="unet_in")

    # Encoder path
    c1, p1 = _enc(inp,  32)
    c2, p2 = _enc(p1,   64)
    c3, p3 = _enc(p2,  128)

    # Bottleneck
    bn = _cbn(p3, 256); bn = _cbn(bn, 256); bn = Dropout(0.3)(bn)

    # Decoder path (produces segmented map)
    d1 = _dec(bn, c3, 128)
    d2 = _dec(d1, c2,  64)
    d3 = _dec(d2, c1,  32)

    # Outputs
    seg_map  = Conv2D(1, 1, activation="sigmoid", name="seg_map")(d3)
    seg_feat = GlobalAveragePooling2D(name="seg_features")(bn)   # 256-dim

    return Model(inp, [seg_map, seg_feat], name="UNet")


def get_unet_feature_extractor() -> Model:
    unet = build_unet((128, 128, 1))
    return Model(unet.input,
                 unet.get_layer("seg_features").output,
                 name="UNet_FeatExtractor")


print("=" * 60)
print("  CELL 7 -- Running Completed [OK]  U-Net defined.")
_u = build_unet()
print(f"  U-Net parameters : {_u.count_params():,}")
print("  Outputs          : seg_map (128x128x1) + seg_features (256-dim)")
del _u; tf.keras.backend.clear_session()
print("=" * 60)


In [ ]:
# ==============================================================================
# CELL 8 -- LTCN: Long-Term Convolutional Network (DiaRetULS-Net Block 2)
#
#   Input: concatenated features (DWT+LBP hand-crafted + U-Net bottleneck)
#   Spatial branch : dense layers (CNN-style)
#   Temporal branch: LSTM layers (capture feature-sequence dependencies)
#   Merge -> st_features (64-dim) -> softmax head
#   During SVM classification, only st_features are extracted.
# ==============================================================================

def build_ltcn(input_dim: int, num_classes: int = NUM_CLASSES) -> Model:
    inp = Input(shape=(input_dim,), name="ltcn_in")

    # -- Spatial branch (dense layers mimic 1-D convolution) ------------------
    x = Dense(512, activation="relu", kernel_regularizer=l2(1e-4))(inp)
    x = BatchNormalization()(x)
    x = Dropout(0.40)(x)
    x = Dense(256, activation="relu", kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.35)(x)

    # -- Temporal branch (LSTM on reshaped spatial features) ------------------
    SEQ, DIM = 16, 16           # reshape 256 -> (16 steps x 16 dim)
    t = Reshape((SEQ, DIM))(x)
    t = LSTM(128, return_sequences=True,  kernel_regularizer=l2(1e-4))(t)
    t = LSTM(64,  return_sequences=False, kernel_regularizer=l2(1e-4))(t)

    # -- Merge spatial + temporal ------------------------------------------
    m  = concatenate([x, t])
    m  = Dense(128, activation="relu", kernel_regularizer=l2(1e-4))(m)
    m  = Dropout(0.25)(m)
    st = Dense(64, activation="relu", name="st_features")(m)  # -> SVM input

    out = Dense(num_classes, activation="softmax", name="predictions")(st)

    model = Model(inp, out, name="LTCN")
    model.compile(
        optimizer=Adam(learning_rate=1e-3, clipnorm=1.0),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


print("=" * 60)
print("  CELL 8 -- Running Completed [OK]  LTCN defined.")
_t = build_ltcn(300)
print(f"  LTCN parameters  : {_t.count_params():,}")
print("  st_features dim  : 64  (-> Multi-Class SVM input)")
del _t; tf.keras.backend.clear_session()
print("=" * 60)


In [ ]:
# ==============================================================================
# CELL 9 -- Multi-Class SVM (DiaRetULS-Net Final Classifier)
#
#   st_features (64-dim, from LTCN) -> StandardScaler -> SVC (RBF) -> Grade 0-4
# ==============================================================================

def build_svm(C=15.0, kernel="rbf", gamma="scale") -> Pipeline:
    return Pipeline([
        ("scaler", StandardScaler()),
        ("svm",    SVC(
            C=C, kernel=kernel, gamma=gamma,
            probability=True,
            decision_function_shape="ovr",
            class_weight="balanced",
            random_state=42,
            cache_size=2000,
        )),
    ])

print("=" * 60)
print("  CELL 9 -- Running Completed [OK]  Multi-Class SVM pipeline defined.")
print("  Kernel: RBF  |  C=15  |  class_weight=balanced  |  OvR multi-class")
print("=" * 60)


In [ ]:
# ==============================================================================
# CELL 10 -- Metrics: Accuracy, Precision, Sensitivity, Specificity, F1
# ==============================================================================

def macro_specificity(y_true, y_pred) -> float:
    classes = np.unique(y_true)
    specs   = []
    for c in classes:
        tn = int(np.sum((y_true != c) & (y_pred != c)))
        fp = int(np.sum((y_true != c) & (y_pred == c)))
        specs.append(tn / (tn + fp + 1e-9))
    return float(np.mean(specs))


def compute_metrics(y_true, y_pred) -> dict:
    return {
        "accuracy"   : round(float(accuracy_score(y_true, y_pred)) * 100, 4),
        "precision"  : round(float(precision_score(y_true, y_pred,
                              average="macro", zero_division=0)) * 100, 4),
        "sensitivity": round(float(recall_score(y_true, y_pred,
                              average="macro", zero_division=0)) * 100, 4),
        "specificity": round(macro_specificity(y_true, y_pred) * 100, 4),
        "f1_score"   : round(float(f1_score(y_true, y_pred,
                              average="macro", zero_division=0)) * 100, 4),
    }


def print_metrics(m: dict, label=""):
    print(f"\n  -- Metrics {('| '+label) if label else ''} --")
    for k, v in m.items():
        bar = "#" * int(v // 5)
        print(f"  {k:<14}: {v:>8.4f}%  {bar}")


print("=" * 60)
print("  CELL 10 -- Running Completed [OK]  Metrics functions ready.")
print("=" * 60)


In [ ]:
# ==============================================================================
# CELL 11 -- Plotting Utilities
# ==============================================================================

_GRADE_CLR  = ["#4CAF50","#2196F3","#FF9800","#F44336","#9C27B0"]
_METRIC_CLR = {
    "accuracy"   : "#1565C0",
    "precision"  : "#2E7D32",
    "sensitivity": "#E65100",
    "specificity": "#6A1B9A",
    "f1_score"   : "#AD1457",
}


def plot_confusion_matrix(y_true, y_pred, dataset_name, run_id, out_dir):
    cm      = confusion_matrix(y_true, y_pred)
    present = sorted(set(list(y_true) + list(y_pred)))
    labels  = [CLASS_NAMES[i] for i in present]

    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels, ax=ax,
                linewidths=0.4, linecolor="gray",
                annot_kws={"fontsize": 9})
    ax.set_xlabel("Predicted Label", fontsize=11)
    ax.set_ylabel("True Label",      fontsize=11)
    ax.set_title(f"{dataset_name}  Confusion Matrix  (Run {run_id})",
                 fontsize=12, fontweight="bold")
    plt.xticks(rotation=30, ha="right", fontsize=8)
    plt.yticks(rotation=0,  fontsize=8)
    plt.tight_layout()
    p = out_dir / f"confusion_matrix_run{run_id:02d}.png"
    fig.savefig(p, dpi=150); plt.close(fig)
    return p


def plot_roc_curves(y_true, y_prob, dataset_name, run_id, out_dir):
    present = sorted(np.unique(y_true))
    y_bin   = label_binarize(y_true, classes=list(range(NUM_CLASSES)))

    fig, ax = plt.subplots(figsize=(9, 7))
    for i in present:
        if i >= y_prob.shape[1]:
            continue
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        ax.plot(fpr, tpr, color=_GRADE_CLR[i], lw=2,
                label=f"{CLASS_NAMES[i]}  AUC={auc(fpr,tpr):.3f}")
    ax.plot([0,1],[0,1],"k--", lw=1)
    ax.set_xlabel("False Positive Rate", fontsize=11)
    ax.set_ylabel("True Positive Rate",  fontsize=11)
    ax.set_title(f"{dataset_name}  ROC Curves  (Run {run_id})",
                 fontsize=12, fontweight="bold")
    ax.legend(loc="lower right", fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    p = out_dir / f"roc_curve_run{run_id:02d}.png"
    fig.savefig(p, dpi=150); plt.close(fig)
    return p


def plot_metrics_bar(metrics, dataset_name, run_id, out_dir):
    keys = list(metrics.keys())
    vals = [metrics[k] for k in keys]
    clrs = [_METRIC_CLR[k] for k in keys]

    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(keys, vals, color=clrs, alpha=0.88,
                  edgecolor="black", linewidth=0.6)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15,
                f"{v:.2f}%", ha="center", va="bottom",
                fontsize=9, fontweight="bold")
    ax.set_ylim(80, 103)
    ax.set_ylabel("Score (%)", fontsize=11)
    ax.set_title(f"{dataset_name}  Metrics  (Run {run_id})",
                 fontsize=12, fontweight="bold")
    ax.axhline(95, color="red", linestyle="--", lw=1.4, alpha=0.7,
               label="95% threshold")
    ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.35)
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    p = out_dir / f"metrics_bar_run{run_id:02d}.png"
    fig.savefig(p, dpi=150); plt.close(fig)
    return p


def plot_run_history(history: dict, dataset_name, out_dir):
    keys = METRIC_KEYS
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes_flat = axes.flatten()
    for ax, key in zip(axes_flat, keys):
        vals = history.get(key, [])
        runs = list(range(1, len(vals)+1))
        clr  = _METRIC_CLR[key]
        ax.plot(runs, vals, color=clr, marker="o", lw=2, markersize=6, alpha=0.85)
        mean = np.mean(vals)
        ax.axhline(mean, color="black", ls="--", lw=1.5, label=f"Mean={mean:.2f}%")
        ax.axhline(95,   color="red",   ls=":",  lw=1.0, alpha=0.7, label="95%")
        ax.set_title(key.replace("_"," ").title(), fontsize=11, fontweight="bold")
        ax.set_xlabel("Run #"); ax.set_ylabel("Score (%)")
        ax.legend(fontsize=8); ax.grid(alpha=0.3)
        ax.set_xticks(runs)
    axes_flat[-1].axis("off")
    fig.suptitle(f"{dataset_name} -- Metrics Across {TOTAL_RUNS} Runs",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    p = out_dir / "run_history.png"
    fig.savefig(p, dpi=150); plt.close(fig)
    return p


def plot_average_metrics(avg, std, dataset_name, out_dir):
    keys = METRIC_KEYS
    vals = [avg[k] for k in keys]
    errs = [std[k] for k in keys]
    clrs = [_METRIC_CLR[k] for k in keys]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.bar(keys, vals, yerr=errs, color=clrs, alpha=0.88,
                  edgecolor="black", linewidth=0.6, capsize=5,
                  error_kw={"elinewidth":2,"ecolor":"black"})
    for bar, v, e in zip(bars, vals, errs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + e + 0.3,
                f"{v:.2f}+-{e:.2f}%", ha="center", va="bottom",
                fontsize=8, fontweight="bold")
    ax.set_ylim(85, 106)
    ax.set_ylabel("Score (%)", fontsize=12)
    ax.set_title(f"{dataset_name} -- Average Metrics ({TOTAL_RUNS} Runs)",
                 fontsize=13, fontweight="bold")
    ax.axhline(95, color="red", ls="--", lw=1.5, alpha=0.7, label="95% threshold")
    rng = TARGET_RANGES[dataset_name]
    ax.axhspan(rng["low"], rng["high"], alpha=0.08, color="green",
               label=f"Target [{rng['low']}-{rng['high']}%]")
    ax.legend(fontsize=10); ax.grid(axis="y", alpha=0.3)
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    p = out_dir / "average_metrics.png"
    fig.savefig(p, dpi=150); plt.close(fig)
    return p


def plot_cross_dataset_comparison(all_avg, all_std, out_dir):
    ds_list = list(all_avg.keys())
    ds_clrs = {"Messidor-2":"#2196F3","APTOS-2019":"#FF9800","IDRiD":"#4CAF50"}
    x       = np.arange(len(METRIC_KEYS))
    width   = 0.25

    fig, ax = plt.subplots(figsize=(15, 7))
    for i, ds in enumerate(ds_list):
        vals = [all_avg[ds][k] for k in METRIC_KEYS]
        errs = [all_std[ds][k] for k in METRIC_KEYS]
        clr  = ds_clrs.get(ds, "gray")
        bars = ax.bar(x + i*width, vals, width,
                      yerr=errs, label=ds, color=clr,
                      alpha=0.85, edgecolor="black", lw=0.6,
                      capsize=4, error_kw={"elinewidth":1.5})
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                    f"{v:.1f}", ha="center", va="bottom", fontsize=7, rotation=45)

    ax.set_xticks(x + width)
    ax.set_xticklabels([k.replace("_"," ").title() for k in METRIC_KEYS], fontsize=11)
    ax.set_ylabel("Score (%)", fontsize=12)
    ax.set_ylim(85, 105)
    ax.set_title(f"DiaRetULS-Net -- Average Metrics All Datasets ({TOTAL_RUNS}-Run)",
                 fontsize=13, fontweight="bold")
    ax.axhline(95, color="red", ls="--", lw=1.5, alpha=0.7, label="95% threshold")
    ax.legend(fontsize=11); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    p = out_dir / "cross_dataset_comparison.png"
    fig.savefig(p, dpi=150); plt.close(fig)
    return p


print("=" * 60)
print("  CELL 11 -- Running Completed [OK]  All plotting utilities ready.")
print("=" * 60)


In [ ]:
# ==============================================================================
# CELL 12 -- CSV Utilities
# ==============================================================================

def save_run_to_csv(metrics: dict, dataset_name: str, run_id: int) -> Path:
    out_dir  = RESULTS_ROOT / dataset_name / "runs"
    out_dir.mkdir(parents=True, exist_ok=True)
    csv_path = out_dir / f"{dataset_name}_all_runs.csv"
    row = {"run": run_id, "dataset": dataset_name,
           "timestamp": datetime.now().isoformat(timespec="seconds")}
    row.update(metrics)
    df_new = pd.DataFrame([row])
    if csv_path.is_file():
        df_new.to_csv(csv_path, mode="a", header=False, index=False)
    else:
        df_new.to_csv(csv_path, index=False)
    return csv_path


def save_average_csv(avg: dict, std: dict,
                     all_runs: list, dataset_name: str) -> Path:
    ds_dir = RESULTS_ROOT / dataset_name
    ds_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for run_id, m in enumerate(all_runs, 1):
        r = {"run": run_id, "dataset": dataset_name}; r.update(m)
        rows.append(r)
    avg_row = {"run": "AVERAGE", "dataset": dataset_name}
    std_row = {"run": "STD_DEV",  "dataset": dataset_name}
    for k in METRIC_KEYS:
        avg_row[k] = round(avg[k], 4)
        std_row[k] = round(std[k], 4)
    rows += [avg_row, std_row]
    df = pd.DataFrame(rows)
    csv_path = ds_dir / f"{dataset_name}_average_metrics.csv"
    df.to_csv(csv_path, index=False)
    print(f"  Average metrics CSV saved: {csv_path.name}")
    return csv_path


def save_global_summary(all_avg: dict, all_std: dict) -> Path:
    rows = []
    for ds in all_avg:
        r = {"dataset": ds}
        for k in METRIC_KEYS:
            r[k]          = round(all_avg[ds][k], 4)
            r[k + "_std"] = round(all_std[ds][k], 4)
        rows.append(r)
    df  = pd.DataFrame(rows)
    csv = RESULTS_ROOT / "global_summary.csv"
    df.to_csv(csv, index=False)
    print(f"  Global summary CSV saved: {csv.name}")
    return csv


print("=" * 60)
print("  CELL 12 -- Running Completed [OK]  CSV utilities ready.")
print("=" * 60)


In [ ]:
# ==============================================================================
# CELL 13 -- Single-Run Full Pipeline
#
#   features + images
#     -> 70/15/15 stratified split
#     -> U-Net   (128x128 patch -> 256-dim bottleneck features)
#     -> Concatenate with DWT+LBP features
#     -> LTCN    (spatial+temporal learning -> st_features 64-dim)
#     -> Multi-Class SVM  (RBF kernel -> Grade 0-4)
#     -> Metrics + Confusion Matrix + ROC + Bar Chart + CSV row
# ==============================================================================

records_cache = {}   # populated in run_dataset()


def run_one_experiment(features, labels, images, dataset_name, run_id) -> dict:
    seed = BASE_SEED + run_id * 13
    np.random.seed(seed); tf.random.set_seed(seed); random.seed(seed)

    run_dir = RESULTS_ROOT / dataset_name / "runs" / f"run_{run_id:02d}"
    run_dir.mkdir(parents=True, exist_ok=True)

    # -- 1. Stratified 70/15/15 split ----------------------------------------
    X_tv, X_te, y_tv, y_te, img_tv, img_te = train_test_split(
        features, labels, images,
        test_size=0.15, random_state=seed, stratify=labels)
    X_tr, X_val, y_tr, y_val, img_tr, img_val = train_test_split(
        X_tv, y_tv, img_tv,
        test_size=(0.15 / 0.85), random_state=seed, stratify=y_tv)

    # -- 2. U-Net feature extraction (spatial encoding) -----------------------
    def _to_unet_input(imgs):
        patches = np.stack([
            cv2.resize(im, PATCH_SIZE)[..., np.newaxis]
            for im in imgs
        ]).astype(np.float32)
        return patches   # (N, 128, 128, 1)

    unet_fe  = get_unet_feature_extractor()
    unet_tr  = unet_fe.predict(_to_unet_input(img_tr),  batch_size=8, verbose=0)
    unet_val = unet_fe.predict(_to_unet_input(img_val), batch_size=8, verbose=0)
    unet_te  = unet_fe.predict(_to_unet_input(img_te),  batch_size=8, verbose=0)
    del unet_fe; tf.keras.backend.clear_session()

    n_tr  = min(len(X_tr),  len(unet_tr))
    n_val = min(len(X_val), len(unet_val))
    n_te  = min(len(X_te),  len(unet_te))

    # Concatenate DWT+LBP (hand-crafted) + U-Net (learned spatial) features
    F_tr  = np.hstack([X_tr[:n_tr],   unet_tr[:n_tr]])
    F_val = np.hstack([X_val[:n_val], unet_val[:n_val]])
    F_te  = np.hstack([X_te[:n_te],   unet_te[:n_te]])
    y_tr2, y_val2, y_te2 = y_tr[:n_tr], y_val[:n_val], y_te[:n_te]

    # -- 3. LTCN spatial-temporal feature learning ----------------------------
    ltcn = build_ltcn(F_tr.shape[1])

    # Class weights for imbalanced data
    cnts = np.bincount(y_tr2, minlength=NUM_CLASSES).astype(float)
    cw   = {c: len(y_tr2) / (NUM_CLASSES * max(cnts[c], 1.0))
            for c in range(NUM_CLASSES)}

    cbs = [
        EarlyStopping(monitor="val_accuracy", patience=10,
                      restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor="val_loss", factor=0.4,
                          patience=5, min_lr=1e-6, verbose=0),
    ]

    ltcn.fit(
        F_tr, y_tr2,
        validation_data=(F_val, y_val2),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=cw,
        callbacks=cbs,
        verbose=0,
    )

    # Extract st_features (64-dim) for Multi-Class SVM
    st_extractor = Model(ltcn.input, ltcn.get_layer("st_features").output)
    ST_tr = st_extractor.predict(F_tr, verbose=0)
    ST_te = st_extractor.predict(F_te, verbose=0)
    del ltcn, st_extractor; tf.keras.backend.clear_session()

    # -- 4. Multi-Class SVM ---------------------------------------------------
    svm = build_svm()
    svm.fit(ST_tr, y_tr2)
    y_pred = svm.predict(ST_te)
    y_prob = svm.predict_proba(ST_te)

    # -- 5. Metrics + Plots + CSV ---------------------------------------------
    metrics = compute_metrics(y_te2, y_pred)

    plot_confusion_matrix(y_te2, y_pred, dataset_name, run_id, run_dir)
    plot_roc_curves(y_te2, y_prob, dataset_name, run_id, run_dir)
    plot_metrics_bar(metrics, dataset_name, run_id, run_dir)

    # Show preprocessing visualisation once per run (1 sample)
    sample_recs = records_cache.get(dataset_name, [])
    if sample_recs:
        visualise_preprocessing_steps(sample_recs[0][0], dataset_name, run_id)

    save_run_to_csv(metrics, dataset_name, run_id)
    return metrics


print("=" * 60)
print("  CELL 13 -- Running Completed [OK]  Single-run pipeline ready.")
print("  Flow: split -> U-Net feats -> LTCN st_feats -> SVM -> metrics")
print("=" * 60)


In [ ]:
# ==============================================================================
# CELL 14 -- Per-Dataset 10-Run Experiment
# ==============================================================================

def run_dataset(dataset_name: str) -> Optional[dict]:
    print(f"\n{'='*65}")
    print(f"  DATASET : {dataset_name}")
    print(f"{'='*65}")

    ds_dir = RESULTS_ROOT / dataset_name
    ds_dir.mkdir(parents=True, exist_ok=True)

    # Load records
    records = load_records(dataset_name)
    if not records:
        print(f"  [ERROR] No images found. Skipping.")
        return None
    records_cache[dataset_name] = records

    # Distribution pie charts
    print("  Generating dataset distribution pie charts ...")
    plot_distribution_piecharts(records, dataset_name)

    # Preprocess: Bilinear -> Green Channel -> CLAHE
    print("  Pre-processing images: Bilinear -> Green Channel -> CLAHE ...")
    images, labels = preprocess_all(records, dataset_name)

    u, c = np.unique(labels, return_counts=True)
    print(f"  Class distribution: { {int(k): int(v) for k,v in zip(u,c)} }")

    # Feature extraction: DWT (edge) + LBP (texture)
    print("  Extracting DWT (edge) + LBP (texture) features ...")
    features = extract_combined_features(images, dataset_name)

    # 10-Run loop
    all_run_metrics = []
    history         = defaultdict(list)

    for run_id in range(1, TOTAL_RUNS + 1):
        print(f"\n  -- Run {run_id:02d}/{TOTAL_RUNS} [{dataset_name}] --------")
        m = run_one_experiment(features, labels, images,
                               dataset_name, run_id)
        all_run_metrics.append(m)
        for k in METRIC_KEYS:
            history[k].append(m[k])
        print_metrics(m, f"{dataset_name} | Run {run_id}")

    # Averages
    avg_m = {k: round(float(np.mean(history[k])), 4) for k in METRIC_KEYS}
    std_m = {k: round(float(np.std(history[k])),  4) for k in METRIC_KEYS}

    # Save CSVs
    save_average_csv(avg_m, std_m, all_run_metrics, dataset_name)

    # Save charts
    plot_average_metrics(avg_m, std_m, dataset_name, ds_dir)
    plot_run_history(dict(history), dataset_name, ds_dir)

    # Final console summary
    print(f"\n  == {dataset_name}  {TOTAL_RUNS}-Run AVERAGE ==")
    print(f"  {'Metric':<16} {'Mean':>9}  {'Std':>8}")
    print(f"  {'-'*37}")
    rng = TARGET_RANGES[dataset_name]
    for k in METRIC_KEYS:
        flag = "[OK]" if avg_m[k] >= rng["low"] else "[!] CHECK"
        print(f"  {k:<16} {avg_m[k]:>8.4f}%  +-{std_m[k]:>6.4f}%  {flag}")

    all_pass = all(avg_m[k] >= rng["low"] for k in METRIC_KEYS)
    print(f"\n  Target range [{rng['low']}-{rng['high']}%]: "
          f"{'ALL PASS [OK]' if all_pass else 'CHECK [!]'}")
    print(f"  Results saved to: results/{dataset_name}/")
    print("=" * 65)

    return {
        "avg"      : avg_m,
        "std"      : std_m,
        "history"  : dict(history),
        "all_runs" : all_run_metrics,
    }


print("=" * 60)
print("  CELL 14 -- Running Completed [OK]  Per-dataset runner ready.")
print("=" * 60)


In [ ]:
# ==============================================================================
# CELL 15 -- Run All Three Datasets
# ==============================================================================

ALL_RESULTS = {}

for _ds in ["Messidor-2", "APTOS-2019", "IDRiD"]:
    _res = run_dataset(_ds)
    if _res is not None:
        ALL_RESULTS[_ds] = _res

print("\n" + "="*65)
print("  CELL 15 -- Running Completed [OK]  All datasets processed.")
print("="*65)
for _ds, _r in ALL_RESULTS.items():
    print(f"  {_ds:<16}  Avg Accuracy = {_r['avg']['accuracy']:.4f}%")
print("="*65)


In [ ]:
# ==============================================================================
# CELL 16 -- Global Summary + Cross-Dataset Comparison Plots
# ==============================================================================

if not ALL_RESULTS:
    print("[WARN] No results. Check dataset paths and re-run.")
else:
    all_avg = {ds: ALL_RESULTS[ds]["avg"] for ds in ALL_RESULTS}
    all_std = {ds: ALL_RESULTS[ds]["std"] for ds in ALL_RESULTS}

    # Global CSV
    save_global_summary(all_avg, all_std)

    # Cross-dataset bar chart
    plot_cross_dataset_comparison(all_avg, all_std, RESULTS_ROOT)

    # Convergence line chart (all metrics, all datasets)
    ds_clrs = {"Messidor-2":"#2196F3","APTOS-2019":"#FF9800","IDRiD":"#4CAF50"}
    fig, axes = plt.subplots(2, 3, figsize=(20, 11))
    axes_flat = axes.flatten()

    for idx, key in enumerate(METRIC_KEYS):
        ax = axes_flat[idx]
        for ds, res in ALL_RESULTS.items():
            hist = res["history"].get(key, [])
            runs = list(range(1, len(hist)+1))
            ax.plot(runs, hist, color=ds_clrs.get(ds,"gray"), alpha=0.3, lw=1)
            ma = pd.Series(hist).rolling(3, min_periods=1).mean().tolist()
            ax.plot(runs, ma, color=ds_clrs.get(ds,"gray"), lw=2.5, label=ds)
        ax.axhline(95, color="red", ls=":", lw=1, alpha=0.7)
        ax.set_title(key.replace("_"," ").title(), fontsize=11, fontweight="bold")
        ax.set_xlabel("Run #"); ax.set_ylabel("Score (%)")
        ax.grid(alpha=0.3)
        ax.set_xticks(range(1, TOTAL_RUNS+1))

    axes_flat[0].legend(fontsize=9)
    axes_flat[-1].axis("off")
    fig.suptitle(f"DiaRetULS-Net -- Metric Convergence ({TOTAL_RUNS} Runs)",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    _cp = RESULTS_ROOT / "convergence_all_datasets.png"
    fig.savefig(_cp, dpi=150); plt.close(fig)
    print(f"  Convergence chart saved: {_cp.name}")

    # Final console table
    print("\n" + "="*80)
    print("  FINAL RESULTS -- DiaRetULS-Net (10-Run Average)")
    print("="*80)
    hdr = f"{'Dataset':<18}" + "".join(
        f"{k.replace('_',' ').title():>14}" for k in METRIC_KEYS)
    print(hdr); print("-"*80)
    for ds, avg in all_avg.items():
        row = f"{ds:<18}" + "".join(f"{avg[k]:>13.4f}%" for k in METRIC_KEYS)
        print(row)
    print("="*80)

print("=" * 60)
print("  CELL 16 -- Running Completed [OK]  Global summary done.")
print("=" * 60)


In [ ]:
# ==============================================================================
# CELL 17 -- Output Directory Tree
# ==============================================================================

def _tree(path: Path, prefix: str = "", depth: int = 0, max_d: int = 5):
    if depth > max_d or not path.exists():
        return
    items = sorted(path.iterdir())
    for i, item in enumerate(items):
        conn = "\\-- " if i == len(items)-1 else "+-- "
        size = (f"  ({item.stat().st_size/1024:.1f} KB)"
                if item.is_file() else "")
        print(prefix + conn + item.name + size)
        if item.is_dir():
            ext = "    " if i == len(items)-1 else "|   "
            _tree(item, prefix+ext, depth+1, max_d)

print(f"\nResults directory: {RESULTS_ROOT}\n")
_tree(RESULTS_ROOT)

print("\n" + "="*60)
print("  CELL 17 -- Running Completed [OK]")
print("  DiaRetULS-Net pipeline complete.")
print("  All CSVs + confusion matrices + ROC + metric charts")
print("  saved under notebooks/DR/results/")
print("="*60)
